In [36]:
# ── 1. Repos & deps ──────────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate huggingface_hub "robomimic<0.3.0" torchvision wrapt pillow pandas diffusers av gymnasium libero
!uv sync

# Patch OAT's lr_scheduler.py: newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean, skipping')

# Patch OAT: заглушить print'ы про нормализацию
for path in [
    'oat/oat/perception/state_encoder.py',
    'oat/oat/perception/robomimic_vision_encoder.py',
]:
    txt = open(path).read()
    txt = txt.replace(
        'print(warning_msg(f"no normalizer params for port {port}, skipping normalization."))',
        'pass  # suppressed'
    ).replace(
        'print(f"no normalizer params for port {port}, skipping normalization.")',
        'pass  # suppressed'
    )
    open(path, 'w').write(txt)
print('normalizer prints suppressed OK')


fatal: destination path 'hnet' already exists and is not an empty directory.
fatal: destination path 'oat' already exists and is not an empty directory.
Cloning into 'congenial-adventure'...
remote: Enumerating objects: 551, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 551 (delta 13), reused 27 (delta 9), pack-reused 513 (from 1)
Receiving objects: 100% (551/551), 81.67 MiB | 40.34 MiB/s, done.
Resolving deltas: 100% (320/320), done.
Filtering content: 100% (2/2), 652.97 MiB | 15.60 MiB/s, done.
Resolved 156 packages in 746ms                                       
⠙ torch==2.11.0                                                                 Audited 153 packages in 82ms
Resolved 156 packages in 1ms
Audited 153 packages in 3ms
lr_scheduler.py already clean, skipping
normalizer prints suppressed OK


In [4]:
!ls -ld /kaggle/working/oat/data/libero/*

-rw-r--r-- 1 root root 3528193175 Apr 13 12:03 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
-rw-r--r-- 1 root root         24 Apr 13 12:03 /kaggle/working/oat/data/libero/README.md


In [2]:
# ── 2. Dataset (скачиваем прямо туда, куда смотрит OAT по дефолту) ───────────
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('hugface')

if hf_token:
    login(token=hf_token)
else:
    print("Ошибка: Секрет 'hugface' не найден.")

In [3]:

from huggingface_hub import snapshot_download
os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/kaggle/working/oat/data/libero'

In [4]:
# Распаковываем архив прямо в ту же папку
!unzip -o -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip -d /kaggle/working/oat/data/libero/

# Проверяем, что папка появилась
!ls -ld /kaggle/working/oat/data/libero/*

drwxrwxr-x 4 root root       4096 Jan 23 06:29 /kaggle/working/oat/data/libero/libero10_N500.zarr
-rw-r--r-- 1 root root 3528193175 Apr 13 23:58 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
drwxr-xr-x 3 root root       4096 Apr 13 23:59 /kaggle/working/oat/data/libero/__MACOSX
-rw-r--r-- 1 root root         24 Apr 13 23:58 /kaggle/working/oat/data/libero/README.md


In [ ]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

In [ ]:

# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
TOK = 'src/model/model.ckpt'
print(f'Using tokenizer: {TOK}')

!HYDRA_FULL_ERROR=1 MPLBACKEND=agg uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt={TOK} \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16


In [20]:
!pip install libero


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 28.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.7/217.7 kB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 kB 14.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.5/217.5 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 9.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [18]:
!uv add av gymnasium libero

Resolved 156 packages in 0.92ms
Audited 153 packages in 2ms


In [ ]:

# ── 7. Evaluation ─────────────────────────────────────────────────────────────
import os, subprocess, pathlib, yaml

os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('PYOPENGL_PLATFORM', None)
os.environ['MPLBACKEND'] = 'agg'   # override Jupyter's inline backend — not available in uv env

subprocess.run(['apt-get', 'install', '-y', '-q', 'libegl1'], check=False)

_cfg_dir = pathlib.Path.home() / '.libero'
_cfg_dir.mkdir(exist_ok=True)
_cfg_file = _cfg_dir / 'config.yaml'

if not _cfg_file.exists():
    _pkg = subprocess.check_output(
        ['uv', 'run', 'python', '-c',
         'import libero.libero, pathlib; print(pathlib.Path(libero.libero.__file__).parent)'],
        text=True
    ).strip()
    _pkg = pathlib.Path(_pkg)
    yaml.dump({
        'benchmark_root': str(_pkg),
        'bddl_files':     str(_pkg / 'bddl_files'),
        'init_states':    str(_pkg / 'init_files'),
        'datasets':       str(_pkg.parent / 'datasets'),
        'assets':         str(_pkg / 'assets'),
    }, _cfg_file.open('w'))
    print(f'Created LIBERO config → {_cfg_file}  (pkg: {_pkg})')
else:
    print(f'LIBERO config already exists: {_cfg_file}')

!MPLBACKEND=agg uv run python scripts/eval_fddrat_libero.py \
    -c src/model/eval_mod.ckpt \
    -o eval_out/ \
    --n_test 50 \
    --n_test_vis 5


Reading package lists...
Building dependency tree...
Reading state information...
libegl1 is already the newest version (1.4.0-1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
LIBERO config already exists: /root/.libero/config.yaml
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /kaggle/working/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The curren

In [ ]:

# ── 8. Results visualization ───────────────────────────────────────────────────
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

log = json.load(open("eval_out/eval_log.json"))

# ── per-task success rates ────────────────────────────────────────────────────
task_keys = sorted([k for k in log if k.endswith("/mean_success_rate_mean")])
task_names = [k.replace("/mean_success_rate_mean", "").split("_SCENE")[0] for k in task_keys]
task_names = [t.replace("LIBERO_", "").replace("_", " ").title() for t in task_names]
task_vals  = [log[k] for k in task_keys]
task_errs  = [log.get(k.replace("_mean", "_stderr"), 0.0) for k in task_keys]

mean_sr  = log.get("mean_success_rate_mean", np.mean(task_vals))
mean_err = log.get("mean_success_rate_stderr", 0.0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("FD-DRAT — LIBERO10 Evaluation", fontsize=14, fontweight="bold")

# Bar chart — per-task
ax = axes[0]
colors = ["#2ecc71" if v >= 0.8 else "#e67e22" if v >= 0.5 else "#e74c3c" for v in task_vals]
bars = ax.barh(task_names, task_vals, xerr=task_errs, color=colors,
               capsize=4, edgecolor="white", height=0.6)
ax.axvline(mean_sr, color="#3498db", linestyle="--", linewidth=1.5,
           label=f"Mean {mean_sr:.1%}")
ax.set_xlim(0, 1.05)
ax.set_xlabel("Success Rate")
ax.set_title("Per-Task Success Rate")
ax.legend(fontsize=9)
legend_patches = [
    mpatches.Patch(color="#2ecc71", label="≥ 80%"),
    mpatches.Patch(color="#e67e22", label="50–80%"),
    mpatches.Patch(color="#e74c3c", label="< 50%"),
]
ax.legend(handles=legend_patches, fontsize=8, loc="lower right")
for bar, val in zip(bars, task_vals):
    ax.text(min(val + 0.01, 1.0), bar.get_y() + bar.get_height() / 2,
            f"{val:.0%}", va="center", fontsize=8)

# Summary panel
ax2 = axes[1]
ax2.axis("off")
lat_p99  = log.get("latency_p99_ms", float("nan"))
lat_mean = log.get("latency_mean_ms", float("nan"))
rows = [
    ["Metric", "Value"],
    ["Mean Success Rate", f"{mean_sr:.1%} ± {mean_err:.1%}"],
    ["Tasks ≥ 80%", f"{sum(v >= 0.8 for v in task_vals)} / {len(task_vals)}"],
    ["Tasks ≥ 50%", f"{sum(v >= 0.5 for v in task_vals)} / {len(task_vals)}"],
    ["Latency mean",  f"{lat_mean:.1f} ms"],
    ["Latency p99",   f"{lat_p99:.1f} ms"],
    ["Num experiments", str(log.get("num_exp", 1))],
]
tbl = ax2.table(cellText=rows[1:], colLabels=rows[0],
                cellLoc="center", loc="center", bbox=[0.05, 0.1, 0.9, 0.8])
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#2c3e50")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#ecf0f1")
ax2.set_title("Summary", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("eval_out/results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nMean success rate: {mean_sr:.1%}  |  p99 latency: {lat_p99:.1f} ms")
